# S8 - Procesamiento en streaming con Spark: ventanas, watermarking y semántica de entrega

**Proyecto Sello — Equipo LLSW3 — Telemetría real de tráfico de red (`uso-red`)**

**Actividad:** pipeline de Spark Structured Streaming sobre `red-telemetria`, el topic de Kafka que S7 deja alimentado con **lecturas reales** de las tarjetas de red de esta PC (no simuladas). Se aplican ventanas de tiempo (fija y deslizante), watermarking, checkpointing, deduplicación acotada, comparación de intervalos de disparo y persistencia en Parquet.

```text
NIC física (Ethernet) ─┐
NIC virtuales ─────────┤  agente/agente_red.py  (Windows, psutil, 1 lectura/s por NIC)
                       ▼  MQTT  lambda26/red/llsw3/telemetria
               mosquitto (contenedor, puerto 41883 del host)
                       ▼
               app/bridge_mqtt_kafka.py → Kafka: red-telemetria (3 particiones, key = sensorId)
                       ▼
               Spark Structured Streaming  ← este notebook (contenedor lambda26-pyspark-red)
               ventanas + watermark + checkpoint + dropDuplicates + Parquet
```

**Contrato del evento `red.lectura` (S7):**

```json
{"tipoEvento": "red.lectura", "sensorId": "Ethernet",
 "bytesSentPerSec": 1234.5, "bytesRecvPerSec": 5678.9,
 "pktsSentPerSec": 10.0, "pktsRecvPerSec": 25.0,
 "errin": 0, "errout": 0, "dropin": 0, "dropout": 0,
 "velocidadEnlaceMbps": 1000, "origen": "agente-windows",
 "timestamp": 1790512778045}
```

## 3.1 Reanudar el entorno y verificar el punto de partida

Desde la raíz del repo (`D:\bigdataproyectosello\s8bigdata`), en PowerShell:

```powershell
# 1. Kafka + Kafka UI (crea la red lambda26-kafka-net)
docker compose -f kafka/compose.yml up -d
docker exec lambda26-kafka /opt/kafka/bin/kafka-topics.sh --describe --topic red-telemetria --bootstrap-server kafka:9092

# 2. mosquitto + contenedor uso-red
docker compose -f uso-red/compose.yml up -d --build

# 3. pyspark: su compose.yml ya declara la red externa lambda26-kafka-net
#    (equivale al compose.kafka.yml de la guía), así que kafka:9092 resuelve
docker compose -f pyspark/compose.yml up -d --build

# 4. Pestañas que quedan corriendo toda la sesión
docker exec -it lambda26-uso-red python /app/bridge_mqtt_kafka.py   # puente MQTT -> Kafka
python -u uso-red/agente/agente_red.py                               # sensor real (NIC)
```

La celda siguiente comprueba, desde **dentro** del contenedor de Spark, que `kafka` resuelve por DNS y que el puerto 9092 responde, y limpia los checkpoints temporales de corridas anteriores para que el notebook sea reproducible.

In [1]:
import socket, shutil, os, datetime, json
print("Hora de ejecución (Lima):", datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=-5))).strftime("%Y-%m-%d %H:%M:%S"))
print("kafka resuelve a:", socket.gethostbyname("kafka"))
with socket.create_connection(("kafka", 9092), timeout=5):
    print("kafka:9092 acepta conexiones")

for d in ["/tmp/s08-chk-sinlimite-red", "/tmp/s08-chk-watermark-red", "/tmp/s08-chk-deslizante-red",
          "/tmp/s08-chk-tardio-red", "/tmp/s08-chk-frio-red", "/tmp/s08-chk-dedup-red"]:
    shutil.rmtree(d, ignore_errors=True)
print("checkpoints temporales limpiados")

Hora de ejecución (Lima): 2026-10-14 17:49:03
kafka resuelve a: 192.168.0.2
kafka:9092 acepta conexiones
checkpoints temporales limpiados


## 3.2 `SparkSession` con el conector de Kafka

`spark.sql.session.timeZone = America/Lima` hace que las columnas de tiempo se muestren en la hora local del equipo, la misma del reloj de Windows en las capturas.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s08-streaming-red-telemetria")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")   # = 3 particiones de red-telemetria
    .config("spark.sql.session.timeZone", "America/Lima")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7bb7c184-6e20-4ca2-8e5b-7e22f2410bc2;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central


	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 212ms :: artifacts dl 6ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org

26/10/14 22:49:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


In [3]:
# Lectura BATCH (no streaming): lo que hay hoy en el topic, y termina
lote = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "red-telemetria")
    .option("startingOffsets", "earliest")
    .load()
)
print("mensajes en red-telemetria:", lote.count())
lote.groupBy("partition", lote.key.cast("string").alias("sensorId")).count().orderBy("partition").show(truncate=False)
lote.selectExpr("CAST(key AS STRING) AS sensorId", "CAST(value AS STRING) AS value", "partition", "offset") \
    .orderBy("timestamp", ascending=False).show(3, truncate=90)

mensajes en red-telemetria: 40727


+---------+----------------------------------+-----+
|partition|sensorId                          |count|
+---------+----------------------------------+-----+
|0        |nic-tardio                        |4    |
|0        |nic-duplicado                     |8    |
|0        |nic-tardio-frio                   |1    |
|0        |desconocido                       |18   |
|0        |Ethernet 2                        |13566|
|2        |Ethernet                          |13567|
|2        |vEthernet (WSL (Hyper-V firewall))|13566|
+---------+----------------------------------+-----+



+----------------------------------+------------------------------------------------------------------------------------------+---------+------+
|                          sensorId|                                                                                     value|partition|offset|
+----------------------------------+------------------------------------------------------------------------------------------+---------+------+
|                        Ethernet 2|{"tipoEvento": "red.lectura", "sensorId": "Ethernet 2", "bytesSentPerSec": 0.0, "bytesR...|        0| 13597|
|vEthernet (WSL (Hyper-V firewall))|{"tipoEvento": "red.lectura", "sensorId": "vEthernet (WSL (Hyper-V firewall))", "bytesS...|        2| 27134|
|                          Ethernet|{"tipoEvento": "red.lectura", "sensorId": "Ethernet", "bytesSentPerSec": 555.2770888679...|        2| 27133|
+----------------------------------+------------------------------------------------------------------------------------------+---

La distribución por partición confirma el particionado por `key` de S7: cada NIC (`sensorId`) cae siempre en la misma partición, lo que garantiza orden por dispositivo.

## 3.3 Leer el topic en modo streaming

In [4]:
crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "red-telemetria")
    .option("startingOffsets", "latest")
    .load()
)
print("es un stream:", crudo.isStreaming)
crudo.printSchema()
try:
    crudo.show()
except Exception as e:
    print("crudo.show() falla como se espera ->", type(e).__name__, ":", str(e).splitlines()[0][:150])

es un stream: True
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

crudo.show() falla como se espera -> AnalysisException : Queries with streaming sources must be executed with writeStream.start(), or from a streaming table or flow definition within a Spark Declarative Pipe


`crudo` es un **plan**, no una tabla: `.show()` lanza `AnalysisException` porque un stream no tiene final que materializar. `startingOffsets="latest"` evita reprocesar las decenas de miles de lecturas históricas en cada consulta de prueba.

## 3.4 Parsear el JSON con un esquema explícito

En streaming el esquema **no se puede inferir** (el stream no termina). El esquema replica el contrato `red.lectura` de S7; `ts` (timestamp de evento, cuando el agente leyó la NIC) es el reloj de todas las ventanas. `uso_enlace_pct` es la métrica de dominio: porcentaje de la capacidad del enlace usado (1 Mbps = 125 000 B/s).

In [5]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema_red = (
    StructType()
    .add("tipoEvento",          StringType())
    .add("sensorId",            StringType())   # NIC: "Ethernet", "Ethernet 2", "vEthernet (...)"
    .add("bytesSentPerSec",     DoubleType())
    .add("bytesRecvPerSec",     DoubleType())
    .add("pktsSentPerSec",      DoubleType())
    .add("pktsRecvPerSec",      DoubleType())
    .add("errin",               LongType())     # contadores acumulados del SO
    .add("errout",              LongType())
    .add("dropin",              LongType())
    .add("dropout",             LongType())
    .add("velocidadEnlaceMbps", DoubleType())
    .add("origen",              StringType())
    .add("timestamp",           LongType())     # epoch en ms
)

eventos = (
    crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_red).alias("e"))
    .select("e.*")
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
    .withColumn("uso_enlace_pct",
                (col("bytesSentPerSec") + col("bytesRecvPerSec")) / (col("velocidadEnlaceMbps") * 125000.0) * 100.0)
)
eventos.printSchema()

root
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- bytesSentPerSec: double (nullable = true)
 |-- bytesRecvPerSec: double (nullable = true)
 |-- pktsSentPerSec: double (nullable = true)
 |-- pktsRecvPerSec: double (nullable = true)
 |-- errin: long (nullable = true)
 |-- errout: long (nullable = true)
 |-- dropin: long (nullable = true)
 |-- dropout: long (nullable = true)
 |-- velocidadEnlaceMbps: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- uso_enlace_pct: double (nullable = true)



## 3.5 Primera consulta: ver los eventos llegar (consola)

Se deja corriendo mientras arranca la consulta a Parquet de 3.6: un mismo DataFrame alimentando dos *sinks*.

In [6]:
consulta = (
    eventos.select("sensorId", "bytesSentPerSec", "bytesRecvPerSec", "pktsRecvPerSec", "uso_enlace_pct", "ts")
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .option("numRows", 9)
    .trigger(processingTime="5 seconds")
    .start()
)
print("consulta de consola:", consulta.id, "activa:", consulta.isActive)

consulta de consola: b7b4febd-5e4d-45ab-9ed8-72d054000af0 activa: True


## 3.6 Persistir en Parquet, con checkpoint — corrida 1

Un *sink* de archivo solo acepta `append`. El checkpoint (`chk_red_parquet_s08`) registra qué offsets ya se escribieron; la carpeta `_spark_metadata` del destino registra qué archivos pertenecen a cada micro-lote.

In [7]:
import time
ARTIFACTS = "/opt/artifacts/s08_red_parquet"
CHECKPOINT_PARQUET = "/opt/artifacts/s08_chk_red_parquet"

def arrancar_parquet():
    return (
        eventos.writeStream
        .outputMode("append")
        .format("parquet")
        .option("path", ARTIFACTS)
        .option("checkpointLocation", CHECKPOINT_PARQUET)
        .trigger(processingTime="5 seconds")
        .start()
    )

def filas_en_parquet():
    try:
        return spark.read.parquet(ARTIFACTS).count()
    except Exception:
        return 0

filas_antes_corrida_1 = filas_en_parquet()
consulta2 = arrancar_parquet()
time.sleep(25)
consulta.stop()       # consola de 3.5
consulta2.stop()      # Parquet de 3.6
filas_corrida_1 = filas_en_parquet()
lotes_corrida_1 = [p["batchId"] for p in consulta2.recentProgress]
print(f"\nCORRIDA 1 -> filas antes: {filas_antes_corrida_1} | filas después: {filas_corrida_1} | micro-lotes: {lotes_corrida_1}")

-------------------------------------------
Batch: 0
-------------------------------------------


+--------+---------------+---------------+--------------+--------------+---+
|sensorId|bytesSentPerSec|bytesRecvPerSec|pktsRecvPerSec|uso_enlace_pct|ts |
+--------+---------------+---------------+--------------+--------------+---+
+--------+---------------+---------------+--------------+--------------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+----------------------------------+------------------+------------------+------------------+---------------------+-----------------------+
|sensorId                          |bytesSentPerSec   |bytesRecvPerSec   |pktsRecvPerSec    |uso_enlace_pct       |ts                     |
+----------------------------------+------------------+------------------+------------------+---------------------+-----------------------+
|Ethernet 2                        |0.0               |0.0               |0.0               |0.0                  |2026-10-14 17:49:13.317|
|Ethernet 2                        |0.0               |0.0               |0.0               |0.0                  |2026-10-14 17:49:14.343|
|Ethernet                          |1195.4238592424401|2492.8960967128933|25.269122227888978|0.0029506559647642667|2026-10-14 17:49:13.317|
|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0     

-------------------------------------------
Batch: 2
-------------------------------------------
+----------------------------------+------------------+------------------+------------------+---------------------+-----------------------+
|sensorId                          |bytesSentPerSec   |bytesRecvPerSec   |pktsRecvPerSec    |uso_enlace_pct       |ts                     |
+----------------------------------+------------------+------------------+------------------+---------------------+-----------------------+
|Ethernet 2                        |0.0               |0.0               |0.0               |0.0                  |2026-10-14 17:49:15.369|
|Ethernet 2                        |0.0               |0.0               |0.0               |0.0                  |2026-10-14 17:49:16.395|
|Ethernet 2                        |0.0               |0.0               |0.0               |0.0                  |2026-10-14 17:49:17.422|
|Ethernet 2                        |0.0               |0.0     

-------------------------------------------
Batch: 3
-------------------------------------------
+----------------------------------+------------------+------------------+------------------+--------------------+-----------------------+
|sensorId                          |bytesSentPerSec   |bytesRecvPerSec   |pktsRecvPerSec    |uso_enlace_pct      |ts                     |
+----------------------------------+------------------+------------------+------------------+--------------------+-----------------------+
|Ethernet 2                        |0.0               |0.0               |0.0               |0.0                 |2026-10-14 17:49:20.5  |
|Ethernet 2                        |0.0               |0.0               |0.0               |0.0                 |2026-10-14 17:49:21.528|
|Ethernet 2                        |0.0               |0.0               |0.0               |0.0                 |2026-10-14 17:49:22.555|
|Ethernet 2                        |0.0               |0.0           

...

-------------------------------------------
Batch: 5
-------------------------------------------
+----------------------------------+------------------+-----------------+------------------+--------------------+-----------------------+
|sensorId                          |bytesSentPerSec   |bytesRecvPerSec  |pktsRecvPerSec    |uso_enlace_pct      |ts                     |
+----------------------------------+------------------+-----------------+------------------+--------------------+-----------------------+
|Ethernet 2                        |0.0               |0.0              |0.0               |0.0                 |2026-10-14 17:49:29.741|
|Ethernet 2                        |0.0               |0.0              |0.0               |0.0                 |2026-10-14 17:49:30.768|
|Ethernet 2                        |0.0               |0.0              |0.0               |0.0                 |2026-10-14 17:49:31.794|
|Ethernet 2                        |0.0               |0.0              |0.

-------------------------------------------
Batch: 6
-------------------------------------------
+----------------------------------+-----------------+-----------------+------------------+---------------------+-----------------------+
|sensorId                          |bytesSentPerSec  |bytesRecvPerSec  |pktsRecvPerSec    |uso_enlace_pct       |ts                     |
+----------------------------------+-----------------+-----------------+------------------+---------------------+-----------------------+
|Ethernet 2                        |0.0              |0.0              |0.0               |0.0                  |2026-10-14 17:49:34.872|
|Ethernet 2                        |0.0              |0.0              |0.0               |0.0                  |2026-10-14 17:49:35.897|
|Ethernet 2                        |0.0              |0.0              |0.0               |0.0                  |2026-10-14 17:49:36.924|
|Ethernet 2                        |0.0              |0.0              |0.0

26/10/14 22:49:40 ERROR FileFormatWriter: Aborting job e1169071-4160-486d-b682-b71a3676ac46.
java.lang.InterruptedException
	[... 45 líneas de traza Java omitidas ...]


26/10/14 22:49:40 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException: Stage cancelled: [SPARK_JOB_CANCELLED] Job 20 cancelled Query [id = 84a29547-b93c-4cfd-a2ba-e7b4de26448f, runId = b0c27b00-154d-46c6-9e81-fcc221c9ed1e] was stopped SQLSTATE: XXKDA
	[... 24 líneas de traza Java omitidas ...]
26/10/14 22:49:40 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException: Stage cancelled: [SPARK_JOB_CANCELLED] Job 20 cancelled Query [id = 84a29547-b93c-4cfd-a2ba-e7b4de26448f, runId = b0c27b00-154d-46c6-9e81-fcc221c9ed1e] was stopped SQLSTATE: XXKDA
	[... 24 líneas de traza Java omitidas ...]



CORRIDA 1 -> filas antes: 156 | filas después: 1195 | micro-lotes: [12, 13, 14, 15, 16]


### Corrida 2 sobre el mismo `path` y el mismo `checkpointLocation`

Si la carpeta se sobreescribiera, el conteo volvería a empezar; si acumula, debe ser **mayor** que el de la corrida 1. Además, el primer `batchId` de la corrida 2 debe continuar la numeración del checkpoint, y su offset de inicio debe ser el offset final de la corrida 1.

In [8]:
time.sleep(5)
consulta2 = arrancar_parquet()
time.sleep(25)
consulta2.stop()
filas_corrida_2 = filas_en_parquet()
prog2 = consulta2.recentProgress
lotes_corrida_2 = [p["batchId"] for p in prog2]
print(f"CORRIDA 2 -> filas después: {filas_corrida_2} | micro-lotes: {lotes_corrida_2}")
print("offset inicial de la corrida 2:", prog2[0]["sources"][0]["startOffset"])
print("¿acumula en vez de sobreescribir?", filas_corrida_2 > filas_corrida_1)

guardado = spark.read.parquet(ARTIFACTS)
guardado.groupBy("sensorId").count().show(truncate=False)
guardado.select("sensorId", "bytesSentPerSec", "bytesRecvPerSec", "uso_enlace_pct", "ts").orderBy(col("ts").desc()).show(6, truncate=False)
print("archivos parquet:", len([f for f in os.listdir(ARTIFACTS) if f.endswith(".parquet")]),
      "| entradas en _spark_metadata:", sorted(os.listdir(f"{ARTIFACTS}/_spark_metadata"), key=lambda x: int(x.split('.')[0]) if x[0].isdigit() else -1)[-3:])

CORRIDA 2 -> filas después: 1297 | micro-lotes: [17, 18, 19, 20, 21, 22]
offset inicial de la corrida 2: {"red-telemetria": {"0": 13620, "1": 0, "2": 27179}}
¿acumula en vez de sobreescribir? True


+----------------------------------+-----+
|sensorId                          |count|
+----------------------------------+-----+
|nic-tardio                        |1    |
|Ethernet                          |431  |
|vEthernet (WSL (Hyper-V firewall))|431  |
|nic-tardio-frio                   |1    |
|nic-duplicado                     |2    |
|Ethernet 2                        |431  |
+----------------------------------+-----+



+----------------------------------+------------------+------------------+---------------------+-----------------------+
|sensorId                          |bytesSentPerSec   |bytesRecvPerSec   |uso_enlace_pct       |ts                     |
+----------------------------------+------------------+------------------+---------------------+-----------------------+
|Ethernet                          |990.0760047275556 |2316.349077989567 |0.0026451400661736983|2026-10-14 17:50:08.734|
|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |2026-10-14 17:50:08.734|
|Ethernet 2                        |0.0               |0.0               |0.0                  |2026-10-14 17:50:08.734|
|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |2026-10-14 17:50:07.708|
|Ethernet                          |1144.3430905479056|2494.3560210494807|0.0029109592892779088|2026-10-14 17:50:07.708|
|Ethernet 2                     

## 3.7 Agregación por ventana fija, **sin** watermark (el problema)

Métrica de dominio: promedio de B/s enviados y recibidos y de `uso_enlace_pct`, por NIC y por ventana de 10 s. Sin watermark solo se admite `complete`, y el estado guarda **todas** las ventanas vistas. Además de la salida por consola, se lee `stateOperators[0].numRowsTotal` del progreso de cada micro-lote: el número de filas que la consulta mantiene en memoria.

In [9]:
from pyspark.sql.functions import window, avg, count

def agregar(df, ancho, paso=None):
    w = window(col("ts"), ancho) if paso is None else window(col("ts"), ancho, paso)
    return df.groupBy(w, col("sensorId")).agg(
        avg("bytesSentPerSec").alias("avg_sent_Bps"),
        avg("bytesRecvPerSec").alias("avg_recv_Bps"),
        avg("uso_enlace_pct").alias("avg_uso_pct"),
        count("*").alias("n"),
    )

def estado_por_lote(q):
    filas = []
    for p in q.recentProgress:
        ops = p.get("stateOperators") or [{}]
        filas.append((p["batchId"], p["numInputRows"], ops[0].get("numRowsTotal"), p.get("eventTime", {}).get("watermark")))
    print(f"{'batchId':>7} | {'filas leídas':>12} | {'filas en estado':>15} | watermark")
    for b, n, s, w in filas:
        print(f"{b:>7} | {n:>12} | {str(s):>15} | {w}")
    return filas

consulta = (
    agregar(eventos, "10 seconds")
    .writeStream.outputMode("complete").format("console")
    .option("truncate", "false").option("numRows", 30)
    .option("checkpointLocation", "/tmp/s08-chk-sinlimite-red")
    .trigger(processingTime="5 seconds")
    .start()
)

In [10]:
time.sleep(40)
consulta.stop()
print("\nESTADO SIN WATERMARK:")
estado_sin_limite = estado_por_lote(consulta)

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+------------+------------+-----------+---+
|window|sensorId|avg_sent_Bps|avg_recv_Bps|avg_uso_pct|n  |
+------+--------+------------+------------+-----------+---+
+------+--------+------------+------------+-----------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+-----------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps     |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+------------------+-----------------+--------------------+---+
|{2026-10-14 17:50:10, 2026-10-14 17:50:20}|Ethernet                          |1857.3300307710554|2510.301611883134|0.003494105314123352|4  |
|{2026-10-14 17:50:10, 2026-10-14 17:50:20}|Ethernet 2                        |0.0               |0.0              |0.0                 |4  |
|{2026-10-14 17:50:10, 2026-10-14 17:50:20}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0              |0.0                 |4  |
+------------------------------------------+-------

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+-----------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps     |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+------------------+-----------------+--------------------+---+
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0              |0.0                 |6  |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|Ethernet 2                        |0.0               |0.0              |0.0                 |6  |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|Ethernet                          |1456.2020391649155|2587.649319839043|0.003235081087203167|6  |
|{2026-10-14 17:50:10, 2026-10-14 17:50:20}|Etherne

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+------------------+------------------+--------------------+---+
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                 |10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|Ethernet 2                        |0.0               |0.0               |0.0                 |10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|Ethernet                          |4239.376434199976 |3168.8203965905113|0.00592655746463239 |10 |
|{2026-10-14 17:50:10, 2026-10-14 17:50:20}|E

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:50:30, 2026-10-14 17:50:40}|vEthernet (WSL (Hyper-V firewall))|0.0               |17.55718415195961 |3.271017075353444E-6 |5  |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|Ethernet 2                        |0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:50:30, 2026-10-14 17:50

-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:50:30, 2026-10-14 17:50:40}|vEthernet (WSL (Hyper-V firewall))|0.0               |8.778592075979805 |1.635508537676722E-6 |10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|Ethernet 2                        |0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:50:30, 2026-10-14 17:50

-------------------------------------------
Batch: 6
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:50:30, 2026-10-14 17:50:40}|vEthernet (WSL (Hyper-V firewall))|0.0               |8.778592075979805 |1.635508537676722E-6 |10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:50:40, 2026-10-14 17:50:50}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |5  |
|{2026-10-14 17:50:20, 2026-10-14 17:50

-------------------------------------------
Batch: 7
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:50:30, 2026-10-14 17:50:40}|vEthernet (WSL (Hyper-V firewall))|0.0               |8.778592075979805 |1.635508537676722E-6 |10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:50:40, 2026-10-14 17:50:50}|vEthernet (WSL (Hyper-V firewall))|9.732193800944422 |9.747061859611117 |3.6291114411840785E-6|9  |
|{2026-10-14 17:50:20, 2026-10-14 17:50

-------------------------------------------
Batch: 8
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:50:30, 2026-10-14 17:50:40}|vEthernet (WSL (Hyper-V firewall))|0.0               |8.778592075979805 |1.635508537676722E-6 |10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50:30}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:50:40, 2026-10-14 17:50:50}|vEthernet (WSL (Hyper-V firewall))|8.758974420849981 |8.772355673650006 |3.2662002970656708E-6|10 |
|{2026-10-14 17:50:20, 2026-10-14 17:50


ESTADO SIN WATERMARK:
batchId | filas leídas | filas en estado | watermark
      0 |            0 |               0 | None
      1 |           12 |               3 | None
      2 |           18 |               6 | None
      3 |           12 |               6 | None
      4 |           15 |               9 | None
      5 |           15 |               9 | None
      6 |           15 |              12 | None
      7 |           12 |              12 | None
      8 |           15 |              15 | None


## 3.8 Watermarking: la misma agregación, acotada

**Margen elegido: 10 segundos.** Justificación para el dominio de tráfico de red:

- El agente publica **1 lectura por segundo por NIC** y la latencia extremo a extremo medida en S7 (NIC → MQTT → bridge → Kafka) fue de ~0,8 s (`latencyMs: 809`). En operación normal ningún evento llega con más de 1–2 s de retraso.
- MQTT con QoS 0 no guarda backlog: si el agente o el bridge se caen, esas lecturas no llegan "tarde", **no llegan**. Un retraso > 10 s solo aparece ante una falla, y en ese caso es preferible no reabrir ventanas que el monitoreo ya reportó.
- 10 s = el ancho de una ventana: el estado se queda en ~2–3 ventanas por NIC (memoria constante), y cada ventana se cierra a más tardar ~20 s después de empezar, suficiente para alertas de saturación casi en tiempo real.
- **Lo que se pierde:** cualquier lectura con más de 10 s de retraso respecto del evento más reciente se descarta en silencio. **Lo que se gana:** memoria acotada y resultados finales en ~20 s.

In [11]:
consulta = (
    agregar(eventos.withWatermark("ts", "10 seconds"), "10 seconds")
    .writeStream.outputMode("update").format("console")
    .option("truncate", "false").option("numRows", 30)
    .option("checkpointLocation", "/tmp/s08-chk-watermark-red")
    .trigger(processingTime="5 seconds")
    .start()
)

In [12]:
time.sleep(40)
consulta.stop()
print("\nESTADO CON WATERMARK DE 10 s:")
estado_watermark = estado_por_lote(consulta)
max_sin = max(s for _, _, s, _ in estado_sin_limite if s is not None)
max_con = max(s for _, _, s, _ in estado_watermark if s is not None)
print(f"\nmáximo de filas en estado -> sin watermark: {max_sin} | con watermark: {max_con}")

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+------------+------------+-----------+---+
|window|sensorId|avg_sent_Bps|avg_recv_Bps|avg_uso_pct|n  |
+------+--------+------------+------------+-----------+---+
+------+--------+------------+------------+-----------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------------------------+-----------------+-----------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps     |avg_recv_Bps     |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+-----------------+-----------------+--------------------+---+
|{2026-10-14 17:50:50, 2026-10-14 17:51:00}|Ethernet 2                        |0.0              |0.0              |0.0                 |2  |
|{2026-10-14 17:50:50, 2026-10-14 17:51:00}|Ethernet                          |3371.887026326628|5697.419547765314|0.007255445259273553|2  |
|{2026-10-14 17:50:50, 2026-10-14 17:51:00}|vEthernet (WSL (Hyper-V firewall))|0.0              |0.0              |0.0                 |2  |
+------------------------------------------+-------------

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+----------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps    |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+----------------+---------------------+---+
|{2026-10-14 17:51:00, 2026-10-14 17:51:10}|vEthernet (WSL (Hyper-V firewall))|17.521014201315534|17.5788670262034|6.539335114582009E-6 |5  |
|{2026-10-14 17:51:00, 2026-10-14 17:51:10}|Ethernet 2                        |0.0               |0.0             |0.0                  |5  |
|{2026-10-14 17:51:00, 2026-10-14 17:51:10}|Ethernet                          |575.9470288617733 |1472.6651296776 |0.0016388897268314988|5  |
+------------------------------------------+-------

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:51:00, 2026-10-14 17:51:10}|vEthernet (WSL (Hyper-V firewall))|8.760507100657767 |8.7894335131017   |3.2696675572910047E-6|10 |
|{2026-10-14 17:51:00, 2026-10-14 17:51:10}|Ethernet 2                        |0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:51:00, 2026-10-14 17:51:10}|Ethernet                          |2935.7355900336106|1896.2244112705212|0.0038655680010433054|10 |
+--------------------------------------

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+----------------------------------+-----------------+------------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps     |avg_recv_Bps      |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+-----------------+------------------+--------------------+---+
|{2026-10-14 17:51:10, 2026-10-14 17:51:20}|Ethernet                          |5439.726227187826|1397.0167650867859|0.005469394393819689|5  |
|{2026-10-14 17:51:10, 2026-10-14 17:51:20}|Ethernet 2                        |0.0              |0.0               |0.0                 |5  |
|{2026-10-14 17:51:10, 2026-10-14 17:51:20}|vEthernet (WSL (Hyper-V firewall))|0.0              |0.0               |0.0                 |5  |
+------------------------------------------+-------

-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+----------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps    |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+------------------+----------------+--------------------+---+
|{2026-10-14 17:51:10, 2026-10-14 17:51:20}|Ethernet                          |3976.8401050627713|940.521068910617|0.003933888939178711|10 |
|{2026-10-14 17:51:10, 2026-10-14 17:51:20}|Ethernet 2                        |0.0               |0.0             |0.0                 |10 |
|{2026-10-14 17:51:10, 2026-10-14 17:51:20}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0             |0.0                 |10 |
+------------------------------------------+-------------

-------------------------------------------
Batch: 6
-------------------------------------------
+------------------------------------------+----------------------------------+-----------------+------------------+-------------------+---+
|window                                    |sensorId                          |avg_sent_Bps     |avg_recv_Bps      |avg_uso_pct        |n  |
+------------------------------------------+----------------------------------+-----------------+------------------+-------------------+---+
|{2026-10-14 17:51:20, 2026-10-14 17:51:30}|Ethernet                          |4429.006865702968|1730.3122566391437|0.00492745529787369|5  |
|{2026-10-14 17:51:20, 2026-10-14 17:51:30}|Ethernet 2                        |0.0              |0.0               |0.0                |5  |
|{2026-10-14 17:51:20, 2026-10-14 17:51:30}|vEthernet (WSL (Hyper-V firewall))|0.0              |0.0               |0.0                |5  |
+------------------------------------------+-------------

-------------------------------------------
Batch: 7
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+-----------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps     |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+------------------+-----------------+--------------------+---+
|{2026-10-14 17:51:20, 2026-10-14 17:51:30}|Ethernet                          |2361.4435168027317|1110.956827711569|0.002777920275611441|10 |
|{2026-10-14 17:51:20, 2026-10-14 17:51:30}|Ethernet 2                        |0.0               |0.0              |0.0                 |10 |
|{2026-10-14 17:51:20, 2026-10-14 17:51:30}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0              |0.0                 |10 |
+------------------------------------------+-------

-------------------------------------------
Batch: 8
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:51:30, 2026-10-14 17:51:40}|Ethernet                          |1193.9013386109077|417.53363667251296|0.0012891479802267364|4  |
|{2026-10-14 17:51:30, 2026-10-14 17:51:40}|Ethernet 2                        |0.0               |0.0               |0.0                  |4  |
|{2026-10-14 17:51:30, 2026-10-14 17:51:40}|vEthernet (WSL (Hyper-V firewall))|21.948266639378193|21.948266639378193|8.178208342572219E-6 |4  |
+--------------------------------------


ESTADO CON WATERMARK DE 10 s:
batchId | filas leídas | filas en estado | watermark
      0 |            0 |               0 | 1970-01-01T00:00:00.000Z
      1 |            6 |               3 | 1970-01-01T00:00:00.000Z
      2 |           15 |               6 | 2026-10-14T22:50:49.032Z
      3 |           15 |               6 | 2026-10-14T22:50:54.167Z
      4 |           15 |               9 | 2026-10-14T22:50:59.320Z
      5 |           15 |               6 | 2026-10-14T22:51:04.451Z
      6 |           15 |               9 | 2026-10-14T22:51:09.586Z
      7 |           15 |               6 | 2026-10-14T22:51:14.717Z
      8 |           12 |               9 | 2026-10-14T22:51:19.850Z

máximo de filas en estado -> sin watermark: 15 | con watermark: 9


## 3.9 Ventana deslizante (15 s, paso 5 s)

Cada lectura cae en **3** ventanas solapadas: suaviza la métrica de uso del enlace y la actualiza cada 5 s, a costa de ~3 veces más estado.

In [13]:
consulta = (
    agregar(eventos.withWatermark("ts", "15 seconds"), "15 seconds", "5 seconds")
    .writeStream.outputMode("update").format("console")
    .option("truncate", "false").option("numRows", 30)
    .option("checkpointLocation", "/tmp/s08-chk-deslizante-red")
    .trigger(processingTime="5 seconds")
    .start()
)
time.sleep(30)
consulta.stop()
print("\nESTADO VENTANA DESLIZANTE:")
estado_deslizante = estado_por_lote(consulta)

-------------------------------------------
Batch: 0
-------------------------------------------
+------+--------+------------+------------+-----------+---+
|window|sensorId|avg_sent_Bps|avg_recv_Bps|avg_uso_pct|n  |
+------+--------+------------+------------+-----------+---+
+------+--------+------------+------------+-----------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+------------------+------------------+--------------------+---+
|{2026-10-14 17:51:40, 2026-10-14 17:51:55}|Ethernet 2                        |0.0               |0.0               |0.0                 |5  |
|{2026-10-14 17:51:35, 2026-10-14 17:51:50}|Ethernet 2                        |0.0               |0.0               |0.0                 |5  |
|{2026-10-14 17:51:30, 2026-10-14 17:51:45}|Ethernet 2                        |0.0               |0.0               |0.0                 |5  |
|{2026-10-14 17:51:30, 2026-10-14 17:51:45}|E

-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:51:40, 2026-10-14 17:51:55}|Ethernet 2                        |0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:51:45, 2026-10-14 17:52:00}|Ethernet 2                        |0.0               |0.0               |0.0                  |5  |
|{2026-10-14 17:51:35, 2026-10-14 17:51:50}|Ethernet 2                        |0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:51:45, 2026-10-14 17:52

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:51:40, 2026-10-14 17:51:55}|Ethernet 2                        |0.0               |0.0               |0.0                  |15 |
|{2026-10-14 17:51:50, 2026-10-14 17:52:05}|Ethernet                          |306.5858718743324 |268.7029321466342 |4.602310432167732E-4 |5  |
|{2026-10-14 17:51:50, 2026-10-14 17:52:05}|vEthernet (WSL (Hyper-V firewall))|41.358528885284684|0.0               |7.705361692647356E-6 |5  |
|{2026-10-14 17:51:45, 2026-10-14 17:52

-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:51:50, 2026-10-14 17:52:05}|Ethernet                          |1628.0601463747305|510.59632677741246|0.0017109251785217144|10 |
|{2026-10-14 17:51:50, 2026-10-14 17:52:05}|vEthernet (WSL (Hyper-V firewall))|82.53569900828032 |0.0               |1.5376935073736435E-5|10 |
|{2026-10-14 17:51:45, 2026-10-14 17:52:00}|Ethernet 2                        |0.0               |0.0               |0.0                  |15 |
|{2026-10-14 17:51:45, 2026-10-14 17:52

-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+-----------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps     |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+-----------------+---------------------+---+
|{2026-10-14 17:51:50, 2026-10-14 17:52:05}|Ethernet                          |1293.4876295781212|514.8836687833615|0.001446697038689186 |14 |
|{2026-10-14 17:51:50, 2026-10-14 17:52:05}|vEthernet (WSL (Hyper-V firewall))|58.95407072020022 |0.0              |1.098352505266888E-5 |14 |
|{2026-10-14 17:52:00, 2026-10-14 17:52:15}|Ethernet 2                        |0.0               |0.0              |0.0                  |4  |
|{2026-10-14 17:51:55, 2026-10-14 17:52:10}|E

-------------------------------------------
Batch: 6
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+-----------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps     |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+-----------------+---------------------+---+
|{2026-10-14 17:51:50, 2026-10-14 17:52:05}|Ethernet                          |1220.2876526780506|495.0242010941734|0.001372249483017779 |15 |
|{2026-10-14 17:51:50, 2026-10-14 17:52:05}|vEthernet (WSL (Hyper-V firewall))|60.88843862116544 |5.864639282311904|1.2436530582855584E-5|15 |
|{2026-10-14 17:52:05, 2026-10-14 17:52:20}|Ethernet                          |236.64061023703968|287.8918898503415|4.1962600006990495E-4|4  |
|{2026-10-14 17:52:00, 2026-10-14 17:52:15}|E


ESTADO VENTANA DESLIZANTE:
batchId | filas leídas | filas en estado | watermark
      0 |            0 |               0 | 1970-01-01T00:00:00.000Z
      1 |           15 |               9 | 1970-01-01T00:00:00.000Z
      2 |           15 |              12 | 2026-10-14T22:51:29.213Z
      3 |           15 |              15 | 2026-10-14T22:51:34.342Z
      4 |           15 |              18 | 2026-10-14T22:51:39.469Z
      5 |           12 |              21 | 2026-10-14T22:51:44.615Z
      6 |           15 |              21 | 2026-10-14T22:51:48.753Z


## 3.10 Checkpointing: qué guarda, y qué pasa al reiniciar

In [14]:
chk = "/tmp/s08-chk-watermark-red"
print("Contenido del checkpoint de 3.8:", sorted(os.listdir(chk)))
print("offsets/ :", sorted(os.listdir(f"{chk}/offsets"), key=lambda x: (len(x), x)))
print("commits/ :", sorted([f for f in os.listdir(f"{chk}/commits") if not f.startswith(".")], key=int))
ultimo = max([f for f in os.listdir(f"{chk}/offsets") if not f.startswith(".")], key=int)
with open(f"{chk}/offsets/{ultimo}") as f:
    lineas = f.read().splitlines()
meta = json.loads(lineas[1])
print(f"\noffsets/{ultimo}:")
print("  version:", lineas[0])
print("  batchWatermarkMs:", meta["batchWatermarkMs"], "| batchTimestampMs:", meta["batchTimestampMs"])
print("  offsets Kafka:", lineas[2])
print("\nstate/ (estado de las ventanas, por operador y partición de shuffle):")
for raiz, dirs, archivos in sorted(os.walk(f"{chk}/state")):
    nivel = raiz.replace(chk, "").count(os.sep)
    visibles = [a for a in archivos if not a.startswith(".")]
    print("  " * nivel + os.path.basename(raiz) + "/", visibles[:6])

Contenido del checkpoint de 3.8: ['.metadata.crc', 'commits', 'metadata', 'offsets', 'sources', 'state']
offsets/ : ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '.0.crc', '.1.crc', '.2.crc', '.3.crc', '.4.crc', '.5.crc', '.6.crc', '.7.crc', '.8.crc', '.9.crc']
commits/ : ['0', '1', '2', '3', '4', '5', '6', '7', '8']

offsets/9:
  version: v1
  batchWatermarkMs: 1792018283952 | batchTimestampMs: 1792018300005
  offsets Kafka: {"red-telemetria":{"0":13742,"1":0,"2":27423}}

state/ (estado de las ventanas, por operador y partición de shuffle):
  state/ []
    0/ []
      0/ ['6.delta', '1.delta', '4.delta', '8.delta', '2.delta', '5.delta']
        _metadata/ ['schema']
      1/ ['6.delta', '1.delta', '4.delta', '8.delta', '2.delta', '5.delta']
      2/ ['6.delta', '1.delta', '4.delta', '8.delta', '2.delta', '5.delta']
      _metadata/ ['metadata']


**Reinicio sobre el mismo checkpoint:** se vuelve a lanzar la consulta de 3.8 con el mismo `checkpointLocation`. Debe continuar desde el `batchId` siguiente al último confirmado y empezar a leer en los offsets guardados, sin reprocesar el topic.

In [15]:
ultimo_batch_previo = estado_watermark[-1][0]
consulta = (
    agregar(eventos.withWatermark("ts", "10 seconds"), "10 seconds")
    .writeStream.outputMode("update").format("console")
    .option("truncate", "false").option("numRows", 30)
    .option("checkpointLocation", chk)
    .trigger(processingTime="5 seconds")
    .start()
)
time.sleep(12)
consulta.stop()
primer = consulta.recentProgress[0]
print(f"\núltimo batchId antes del reinicio: {ultimo_batch_previo} | primer batchId tras el reinicio: {primer['batchId']}")
print("startOffset tras el reinicio:", primer["sources"][0]["startOffset"])
print("offsets guardados en el checkpoint antes del reinicio:", lineas[2])

-------------------------------------------
Batch: 9
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:51:30, 2026-10-14 17:51:40}|Ethernet                          |859.4800274672493 |428.8433373820891 |0.0010306586918794707|9  |
|{2026-10-14 17:51:30, 2026-10-14 17:51:40}|Ethernet 2                        |0.0               |0.0               |0.0                  |9  |
|{2026-10-14 17:51:30, 2026-10-14 17:51:40}|vEthernet (WSL (Hyper-V firewall))|18.846990916760394|18.846990916760394|7.0226328520765325E-6|9  |
+--------------------------------------

-------------------------------------------
Batch: 10
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct          |n  |
+------------------------------------------+----------------------------------+------------------+------------------+---------------------+---+
|{2026-10-14 17:52:10, 2026-10-14 17:52:20}|Ethernet 2                        |0.0               |0.0               |0.0                  |4  |
|{2026-10-14 17:51:40, 2026-10-14 17:51:50}|Ethernet                          |1949.1491523198342|319.8566765101468 |0.0018152046630639852|10 |
|{2026-10-14 17:51:40, 2026-10-14 17:51:50}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                  |10 |
|{2026-10-14 17:51:50, 2026-10-14 17:5

-------------------------------------------
Batch: 11
-------------------------------------------
+------------------------------------------+----------------------------------+------------------+------------------+--------------------+---+
|window                                    |sensorId                          |avg_sent_Bps      |avg_recv_Bps      |avg_uso_pct         |n  |
+------------------------------------------+----------------------------------+------------------+------------------+--------------------+---+
|{2026-10-14 17:52:10, 2026-10-14 17:52:20}|Ethernet 2                        |0.0               |0.0               |0.0                 |9  |
|{2026-10-14 17:52:10, 2026-10-14 17:52:20}|Ethernet                          |1812.1308968012781|1031.7962634959868|0.002275141728237812|9  |
|{2026-10-14 17:52:10, 2026-10-14 17:52:20}|vEthernet (WSL (Hyper-V firewall))|0.0               |0.0               |0.0                 |9  |
+------------------------------------------+


último batchId antes del reinicio: 8 | primer batchId tras el reinicio: 9
startOffset tras el reinicio: {"red-telemetria": {"0": 13737, "1": 0, "2": 27413}}
offsets guardados en el checkpoint antes del reinicio: {"red-telemetria":{"0":13742,"1":0,"2":27423}}


## 3.11 Un dato tardío, y el watermark descartándolo

Se usa el mismo margen justificado (10 s). Primero corre la consulta con datos en vivo ~20 s (para que el watermark avance) y **después** un hilo publica en `red-telemetria` una lectura de la NIC ficticia `nic-tardio` con `timestamp` **45 s en el pasado** y tráfico exagerado (999 999,9 B/s), fácil de reconocer. La salida se imprime por consola **y** se guarda en una lista vía `foreachBatch`, para verificar por código que nunca aparece.

In [16]:
import threading
from pyspark.sql.functions import to_json, struct

COLUMNAS = ["tipoEvento", "sensorId", "bytesSentPerSec", "bytesRecvPerSec", "pktsSentPerSec", "pktsRecvPerSec",
            "errin", "errout", "dropin", "dropout", "velocidadEnlaceMbps", "origen", "timestamp"]

def publicar(sensor_id, bytes_ps, ts_ms, veces=1):
    fila = [("red.lectura", sensor_id, bytes_ps, bytes_ps, 50.0, 50.0, 0, 0, 0, 0, 1000.0, "prueba-s08", ts_ms)]
    df = spark.createDataFrame(fila, COLUMNAS)
    salida = df.select(col("sensorId").alias("key"), to_json(struct(*COLUMNAS)).alias("value"))
    for _ in range(veces):
        salida.write.format("kafka").option("kafka.bootstrap.servers", "kafka:9092").option("topic", "red-telemetria").save()

publicacion_tardio = {}
def publicar_evento_tardio(espera_s, en_el_pasado_s):
    time.sleep(espera_s)
    ahora = int(time.time() * 1000)
    publicacion_tardio.update(publicado_ms=ahora, ts_evento_ms=ahora - en_el_pasado_s * 1000)
    publicar("nic-tardio", 999999.9, ahora - en_el_pasado_s * 1000)
    print(f">>> publicado nic-tardio con timestamp {en_el_pasado_s}s en el pasado")

def colector(destino):
    def f(df, batch_id):
        filas = df.collect()
        destino.extend((batch_id, r["sensorId"], r["window"]["start"], r["n"]) for r in filas)
        print(f"--- Batch {batch_id}: {len(filas)} filas -> {sorted({r['sensorId'] for r in filas})}")
    return f

salida_tardio = []
consulta = (
    eventos.withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId")).agg(count("*").alias("n"))
    .writeStream.outputMode("update")
    .foreachBatch(colector(salida_tardio))
    .option("checkpointLocation", "/tmp/s08-chk-tardio-red")
    .trigger(processingTime="4 seconds")
    .start()
)
hilo = threading.Thread(target=publicar_evento_tardio, args=(20, 45), daemon=True)
hilo.start()

In [17]:
time.sleep(50)
consulta.stop()

# 1) El evento SÍ está en Kafka
en_kafka = (spark.read.format("kafka").option("kafka.bootstrap.servers", "kafka:9092")
            .option("subscribe", "red-telemetria").option("startingOffsets", "earliest").load()
            .filter(col("key").cast("string") == "nic-tardio")
            .filter(from_json(col("value").cast("string"), esquema_red)["timestamp"] == publicacion_tardio["ts_evento_ms"])  # solo esta corrida
            .select(col("key").cast("string").alias("key"), "partition", "offset",
                    col("timestamp").alias("llegada_kafka"),
                    from_json(col("value").cast("string"), esquema_red)["timestamp"].alias("ts_evento_ms")))
en_kafka.show(truncate=False)

# 2) Watermark vigente cuando llegó, y 3) ausencia en la salida
from datetime import datetime
wm = [(p["batchId"], p["numInputRows"], p["eventTime"].get("watermark")) for p in consulta.recentProgress]
for b, n, w in wm:
    print(f"batch {b:>2} | filas {n:>3} | watermark {w}")
ts_ev = datetime.fromtimestamp(publicacion_tardio["ts_evento_ms"] / 1000)
print("\ntimestamp del evento tardío (UTC):", ts_ev.strftime("%H:%M:%S"))
print("veces que nic-tardio aparece en la salida agregada:", sum(1 for f in salida_tardio if f[1] == "nic-tardio"))
def ms(iso): return datetime.fromisoformat(iso.replace("Z", "+00:00")).timestamp() * 1000
wm_al_publicar = [w for p in consulta.recentProgress if ms(p["timestamp"]) <= publicacion_tardio["publicado_ms"] for w in [p["eventTime"].get("watermark")]][-1]
print("watermark vigente al publicarlo:", wm_al_publicar,
      f"-> el evento estaba {(ms(wm_al_publicar) - publicacion_tardio['ts_evento_ms'])/1000:.1f} s por detrás del watermark")
print("micro-lotes procesados después de publicarlo:",
      sum(1 for p in consulta.recentProgress if ms(p["timestamp"]) > publicacion_tardio["publicado_ms"]))

--- Batch 0: 0 filas -> []


--- Batch 1: 3 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


--- Batch 2: 6 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


--- Batch 3: 3 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


--- Batch 4: 3 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


...

--- Batch 10: 3 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


--- Batch 11: 3 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


--- Batch 12: 6 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


--- Batch 13: 3 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


+----------+---------+------+---------------------+-------------+
|key       |partition|offset|llegada_kafka        |ts_evento_ms |
+----------+---------+------+---------------------+-------------+
|nic-tardio|0        |13806 |2026-10-14 17:52:45.8|1792018319971|
+----------+---------+------+---------------------+-------------+

batch  0 | filas   0 | watermark 1970-01-01T00:00:00.000Z
batch  1 | filas  12 | watermark 1970-01-01T00:00:00.000Z
batch  2 | filas  12 | watermark 2026-10-14T22:52:17.363Z
batch  3 | filas  12 | watermark 2026-10-14T22:52:21.464Z
batch  4 | filas   9 | watermark 2026-10-14T22:52:25.569Z
batch  5 | filas  12 | watermark 2026-10-14T22:52:28.645Z
batch  6 | filas  13 | watermark 2026-10-14T22:52:32.748Z
batch  7 | filas  12 | watermark 2026-10-14T22:52:36.861Z
batch  8 | filas  12 | watermark 2026-10-14T22:52:40.993Z
batch  9 | filas  12 | watermark 2026-10-14T22:52:45.094Z
batch 10 | filas  12 | watermark 2026-10-14T22:52:49.217Z
batch 11 | filas   9 | watermar

### Hallazgo: el "arranque en frío" del watermark

Misma prueba, pero publicando el evento tardío **1 s después** de arrancar una consulta nueva (checkpoint vacío): llega en el primer micro-lote con datos, cuando el watermark todavía vale 0 (no hay ningún evento previo contra el cual comparar), y por eso **se acepta**.

In [18]:
salida_frio = []
consulta = (
    eventos.withWatermark("ts", "10 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("sensorId")).agg(count("*").alias("n"))
    .writeStream.outputMode("update")
    .foreachBatch(colector(salida_frio))
    .option("checkpointLocation", "/tmp/s08-chk-frio-red")
    .trigger(processingTime="4 seconds")
    .start()
)
time.sleep(1)
publicar("nic-tardio-frio", 999999.9, int(time.time() * 1000) - 45000)
print(">>> publicado nic-tardio-frio (45 s en el pasado) al inicio de la consulta")
time.sleep(14)
consulta.stop()
for p in consulta.recentProgress:
    print(f"batch {p['batchId']} | filas {p['numInputRows']} | watermark {p['eventTime'].get('watermark')}")
print("filas de nic-tardio-frio en la salida:", [f for f in salida_frio if f[1] == "nic-tardio-frio"])

--- Batch 0: 0 filas -> []


>>> publicado nic-tardio-frio (45 s en el pasado) al inicio de la consulta


--- Batch 1: 4 filas -> ['Ethernet', 'Ethernet 2', 'nic-tardio-frio', 'vEthernet (WSL (Hyper-V firewall))']


--- Batch 2: 6 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


--- Batch 3: 3 filas -> ['Ethernet', 'Ethernet 2', 'vEthernet (WSL (Hyper-V firewall))']


batch 0 | filas 0 | watermark 1970-01-01T00:00:00.000Z
batch 1 | filas 7 | watermark 1970-01-01T00:00:00.000Z
batch 2 | filas 15 | watermark 2026-10-14T22:53:08.746Z
batch 3 | filas 12 | watermark 2026-10-14T22:53:13.875Z
filas de nic-tardio-frio en la salida: [(1, 'nic-tardio-frio', datetime.datetime(2026, 10, 14, 22, 52, 30), 1)]


Exception in thread "stream execution thread for [id = b4cbc22f-4004-4c02-a6b7-01fa414c4747, runId = fbacb954-50e5-44cc-9720-fcbe793f2e93]" java.lang.StackOverflowError
	[... 1024 líneas de traza Java omitidas ...]


## 3.12 Deduplicación acotada por watermark

Caso real de `uso-red`: si el bridge reintenta un `producer.send()` tras un timeout, la misma lectura llega dos veces a Kafka (*at-least-once*). Se publica la lectura `nic-duplicado` **dos veces con el mismo `timestamp`**.

Detalle técnico: la clave de deduplicación incluye `ts`, la **columna del watermark**. Si la clave fuera solo `["sensorId", "timestamp"]` (un `Long`, sin watermark), Spark no podría saber cuándo olvidar una clave y el estado crecería sin límite igual que en 3.7.

In [19]:
salida_dedup = []
def colector_eventos(df, batch_id):
    filas = df.select("sensorId", "bytesSentPerSec", "timestamp").collect()
    salida_dedup.extend((batch_id, r["sensorId"], r["timestamp"]) for r in filas)
    dup = [r for r in filas if r["sensorId"] == "nic-duplicado"]
    print(f"--- Batch {batch_id}: {len(filas)} eventos únicos" + (f" | nic-duplicado: {dup}" if dup else ""))

consulta = (
    eventos.withWatermark("ts", "30 seconds")
    .dropDuplicates(["sensorId", "ts"])
    .writeStream.outputMode("append")
    .foreachBatch(colector_eventos)
    .option("checkpointLocation", "/tmp/s08-chk-dedup-red")
    .trigger(processingTime="3 seconds")
    .start()
)
time.sleep(8)
ts_duplicado = int(time.time() * 1000)
publicar("nic-duplicado", 1234.5, ts_duplicado, veces=2)
print(f">>> nic-duplicado publicado DOS veces con timestamp {ts_duplicado}")

--- Batch 0: 0 eventos únicos


--- Batch 1: 3 eventos únicos


--- Batch 2: 9 eventos únicos


--- Batch 3: 6 eventos únicos


>>> nic-duplicado publicado DOS veces con timestamp 1792018421890


In [20]:
time.sleep(15)
consulta.stop()
veces_kafka = (spark.read.format("kafka").option("kafka.bootstrap.servers", "kafka:9092")
               .option("subscribe", "red-telemetria").option("startingOffsets", "earliest").load()
               .filter(col("key").cast("string") == "nic-duplicado")
               .filter(from_json(col("value").cast("string"), esquema_red)["timestamp"] == ts_duplicado)  # solo esta corrida
               .select("partition", "offset", col("timestamp").alias("llegada_kafka"), col("value").cast("string").alias("value")))
veces_kafka.show(truncate=80)
print("veces en Kafka:", veces_kafka.count(),
      "| veces en la salida deduplicada:", sum(1 for f in salida_dedup if f[1] == "nic-duplicado"))
ops = consulta.lastProgress["stateOperators"][0]
print("operador:", ops["operatorName"], "| claves en estado:", ops["numRowsTotal"], "| duplicados descartados (último lote):", ops.get("customMetrics", {}).get("numDroppedDuplicateRows"))

--- Batch 4: 7 eventos únicos | nic-duplicado: [Row(sensorId='nic-duplicado', bytesSentPerSec=1234.5, timestamp=1792018421890)]


--- Batch 5: 9 eventos únicos


--- Batch 6: 9 eventos únicos


--- Batch 7: 12 eventos únicos


--- Batch 8: 9 eventos únicos


Exception in thread "stream execution thread for [id = 9a66598d-51de-4b29-bf54-b8bdff0e971b, runId = 55707fbc-88c8-4cfe-8584-0d1855198acc]" java.lang.StackOverflowError
	[... 1024 líneas de traza Java omitidas ...]


+---------+------+-----------------------+--------------------------------------------------------------------------------+
|partition|offset|          llegada_kafka|                                                                           value|
+---------+------+-----------------------+--------------------------------------------------------------------------------+
|        0| 13862|2026-10-14 17:53:41.974|{"tipoEvento":"red.lectura","sensorId":"nic-duplicado","bytesSentPerSec":1234...|
|        0| 13863|2026-10-14 17:53:42.049|{"tipoEvento":"red.lectura","sensorId":"nic-duplicado","bytesSentPerSec":1234...|
+---------+------+-----------------------+--------------------------------------------------------------------------------+



veces en Kafka: 2 | veces en la salida deduplicada: 1
operador: dedupe | claves en estado: 64 | duplicados descartados (último lote): 0


## 3.13 Intervalo de disparo: latencia frente a throughput

Cada consulta corre **exactamente 20 s de reloj** (`awaitTermination`). Además del último `batchId`, se mide: filas por micro-lote (throughput por lote) y la latencia real por evento = `processed_at` (`current_timestamp()` del micro-lote) − `ts` (cuando el agente leyó la NIC).

In [21]:
from pyspark.sql.functions import current_timestamp, unix_millis
resultados_trigger = {}

def contar_microlotes(intervalo, segundos_totales):
    nombre = f"conteo_{intervalo.replace(' ', '_')}"
    q = (
        eventos.select("sensorId", "ts", unix_millis("ts").alias("ts_ms"))
        .withColumn("processed_ms", unix_millis(current_timestamp()))
        .writeStream.outputMode("append").format("memory").queryName(nombre)
        .trigger(processingTime=intervalo)
        .start()
    )
    q.awaitTermination(segundos_totales)
    q.stop()
    prog = [p for p in q.recentProgress if p["numInputRows"] > 0]
    lat = spark.sql(f"SELECT avg(processed_ms - ts_ms) AS lat_ms, count(*) AS filas FROM {nombre}").first()
    r = dict(ultimo_batchId=q.lastProgress["batchId"],
             filas=lat["filas"],
             filas_por_lote=round(sum(p["numInputRows"] for p in prog) / max(len(prog), 1), 1),
             latencia_media_ms=round(lat["lat_ms"] or 0),
             duracion_media_lote_ms=round(sum(p["durationMs"]["triggerExecution"] for p in prog) / max(len(prog), 1)))
    resultados_trigger[intervalo] = r
    return r["ultimo_batchId"]

ultimo_lote_rapido = contar_microlotes("1 second", 20)
ultimo_lote_lento = contar_microlotes("6 seconds", 20)
print("Con disparo cada 1 segundo, último batchId en 20s:", ultimo_lote_rapido)
print("Con disparo cada 6 segundos, último batchId en 20s:", ultimo_lote_lento)
print()
for k, v in resultados_trigger.items():
    print(f"{k:>9}: {v}")

26/10/14 22:54:20 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 19, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@4ccd4e0d] is aborting.
26/10/14 22:54:20 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 19, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@4ccd4e0d] aborted.


26/10/14 22:54:20 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException: Stage cancelled: [SPARK_JOB_CANCELLED] Job 123 cancelled Query conteo_1_second [id = 513d1003-f23d-40f0-a4cc-a44237ec1531, runId = 90061591-805f-445e-b9e2-e3efbed9fb71] was stopped SQLSTATE: XXKDA
	[... 29 líneas de traza Java omitidas ...]
26/10/14 22:54:20 ERROR DataWritingSparkTask: Aborting commit for partition 0 (task 406, attempt 0, stage 187.0)
26/10/14 22:54:20 ERROR DataWritingSparkTask: Aborted commit for partition 0 (task 406, attempt 0, stage 187.0)
26/10/14 22:54:20 ERROR Utils: Aborting task
org.apache.spark.SparkRuntimeException: [EXPRESSION_DECODING_FAILED] Failed to decode a row to a value of the expressions: createexternalrow(invoke(input[0, string, true].toString()), static_invoke(DateTimeUtils.toJavaTimestamp(input[1, timestamp, true])), static_invoke(java.lang.Long.valueOf(input[2, bigint, true])), static_invoke(java.lang.Long.valueOf(input[3, bigint, false])), StructField(sensorId,S

Con disparo cada 1 segundo, último batchId en 20s: 18
Con disparo cada 6 segundos, último batchId en 20s: 3

 1 second: {'ultimo_batchId': 18, 'filas': 57, 'filas_por_lote': 3.2, 'latencia_media_ms': 560, 'duracion_media_lote_ms': 633}
6 seconds: {'ultimo_batchId': 3, 'filas': 48, 'filas_por_lote': 16.0, 'latencia_media_ms': 2718, 'duracion_media_lote_ms': 609}


## Errores reales provocados (para el diagnóstico del informe)

1. Escribir una agregación en modo `update` directo a Parquet.
2. Escribir a Parquet **sin** `checkpointLocation`.

In [22]:
for titulo, construir in [
    ("agregación update -> Parquet", lambda: agregar(eventos.withWatermark("ts", "10 seconds"), "10 seconds")
        .writeStream.outputMode("update").format("parquet").option("path", "/tmp/s08-err1")
        .option("checkpointLocation", "/tmp/s08-chk-err1").start()),
    ("Parquet sin checkpointLocation", lambda: eventos.writeStream.format("parquet")
        .option("path", "/tmp/s08-err2").start()),
]:
    try:
        q = construir(); q.stop()
        print(f"[{titulo}] arrancó (inesperado)")
    except Exception as e:
        print(f"[{titulo}] {type(e).__name__}: {str(e).splitlines()[0][:260]}")

[agregación update -> Parquet] AnalysisException: [STREAMING_OUTPUT_MODE.UNSUPPORTED_DATASOURCE] Invalid streaming output mode: update. This output mode is not supported in Data Source parquet. SQLSTATE: 42KDE
[Parquet sin checkpointLocation] AnalysisException: checkpointLocation must be specified either through option("checkpointLocation", ...) or SparkSession.conf.set("spark.sql.streaming.checkpointLocation", ...).


## Resumen de resultados

In [23]:
print(f"Parquet: corrida 1 = {filas_corrida_1} filas, corrida 2 = {filas_corrida_2} filas (acumula: {filas_corrida_2 > filas_corrida_1})")
print(f"Estado máximo: sin watermark = {max_sin} filas | watermark 10 s = {max_con} filas | deslizante = {max(s for _,_,s,_ in estado_deslizante if s is not None)} filas")
print(f"Tardío (45 s, watermark 10 s): apariciones en salida = {sum(1 for f in salida_tardio if f[1]=='nic-tardio')}")
print(f"Arranque en frío: apariciones de nic-tardio-frio = {sum(1 for f in salida_frio if f[1]=='nic-tardio-frio')}")
print(f"Duplicado: publicado 2 veces, aparece {sum(1 for f in salida_dedup if f[1]=='nic-duplicado')} vez")
print(f"Trigger 1 s -> batchId {ultimo_lote_rapido} | Trigger 6 s -> batchId {ultimo_lote_lento}")

Parquet: corrida 1 = 1195 filas, corrida 2 = 1297 filas (acumula: True)
Estado máximo: sin watermark = 15 filas | watermark 10 s = 9 filas | deslizante = 21 filas
Tardío (45 s, watermark 10 s): apariciones en salida = 0
Arranque en frío: apariciones de nic-tardio-frio = 1
Duplicado: publicado 2 veces, aparece 1 vez
Trigger 1 s -> batchId 18 | Trigger 6 s -> batchId 3


## Reflexión técnica

Un sistema que agrega por ventanas no puede esperar para siempre a un dato tardío porque cada ventana abierta vive en el estado de la consulta: sin un límite, el estado crece con cada ventana nueva (3.7) hasta agotar la memoria, y ningún resultado sería nunca definitivo. El watermark convierte esa espera infinita en una regla explícita. En `uso-red` elegí 10 s: el agente publica cada segundo con ~0,8 s de latencia y MQTT QoS 0 no acumula backlog, así que un retraso mayor solo indica una falla. Gano memoria constante (unas pocas ventanas por NIC) y promedios de uso del enlace definitivos en ~20 s, útiles para alertas de saturación; pierdo, en silencio, cualquier lectura con más de 10 s de retraso, que en este dominio prefiero descartar antes que reabrir ventanas ya reportadas.